In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/benhamner/sf-bay-area-bike-share/status.csv
/kaggle/input/datasets/benhamner/sf-bay-area-bike-share/weather.csv
/kaggle/input/datasets/benhamner/sf-bay-area-bike-share/station.csv
/kaggle/input/datasets/benhamner/sf-bay-area-bike-share/trip.csv
/kaggle/input/datasets/benhamner/sf-bay-area-bike-share/database.sqlite


In [14]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("benhamner/sf-bay-area-bike-share")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/benhamner/sf-bay-area-bike-share


In [29]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql import types as T
from geopy.distance import geodesic
from math import sqrt
import pandas as pd
from pyspark.sql.functions import col, sum as Fsum, desc

In [27]:
df_trip = spark.read.csv("/kaggle/input/datasets/benhamner/sf-bay-area-bike-share/trip.csv", header=True, inferSchema=True)
df_stations = spark.read.csv("/kaggle/input/datasets/benhamner/sf-bay-area-bike-share/station.csv", header=True, inferSchema=True)

# Решите следующие задачи для данных велопарковок Сан-Франциско (trips.csv, stations.csv):
1. Найти велосипед с максимальным временем пробега.
2. Найти наибольшее геодезическое расстояние между станциями.
3. Найти путь велосипеда с максимальным временем пробега через станции.
4. Найти количество велосипедов в системе.
5. Найти пользователей потративших на поездки более 3 часов.

In [17]:
spark = SparkSession.builder.getOrCreate()
spark

**Найти велосипед с максимальным временем пробега**

In [24]:
max_duration_bike = (
    df_trip.groupBy("bike_id")
      .agg((Fsum("duration") / 60).alias("total_duration_min"))
      .orderBy(desc("total_duration_min"))
      .limit(1)
)

max_duration_bike.show(truncate=False)

+-------+------------------+
|bike_id|total_duration_min|
+-------+------------------+
|535    |310194.88333333336|
+-------+------------------+



**Найти наибольшее геодезическое расстояние между станциями**

In [30]:
station_coords = df_stations.select("id", "lat", "long")

cross_joined = (
    station_coords.alias("s1")
    .crossJoin(station_coords.alias("s2"))
    .filter(F.col("s1.id") < F.col("s2.id"))
)

def calculate_distance(lat1, lon1, lat2, lon2):
    return geodesic((lat1, lon1), (lat2, lon2)).km

distance_udf = F.udf(calculate_distance, T.DoubleType())

stations_with_distance = cross_joined.withColumn(
    "distance",
    distance_udf(
        F.col("s1.lat"),
        F.col("s1.long"),
        F.col("s2.lat"),
        F.col("s2.long")
    )
)

max_distance = stations_with_distance.orderBy(F.desc("distance")).limit(1)

max_distance.select(
    F.col("s1.id").alias("station1_id"),
    F.col("s2.id").alias("station2_id"),
    F.col("distance").alias("max_distance_km")
).show(truncate=False)

+-----------+-----------+-----------------+
|station1_id|station2_id|max_distance_km  |
+-----------+-----------+-----------------+
|16         |60         |69.92096757764355|
+-----------+-----------+-----------------+



**Найти путь велосипеда с максимальным временем пробега через станции**

In [32]:
row = max_duration_bike.collect()[0]
bike_id = row["bike_id"]

bike_path = (
    df_trip.filter(F.col("bike_id") == bike_id)
    .select("bike_id", "start_station_name", "end_station_name", "start_date", "end_date", "duration")
    .orderBy("start_date")
)

bike_path.select("start_station_name", "end_station_name", "duration").show(truncate=False)

+---------------------------------------------+---------------------------------------------+--------+
|start_station_name                           |end_station_name                             |duration|
+---------------------------------------------+---------------------------------------------+--------+
|Mechanics Plaza (Market at Battery)          |Embarcadero at Sansome                       |3289    |
|Embarcadero at Sansome                       |Market at 4th                                |1286    |
|Market at 4th                                |South Van Ness at Market                     |795     |
|Market at 10th                               |Powell Street BART                           |235     |
|Embarcadero at Folsom                        |San Francisco Caltrain (Townsend at 4th)     |596     |
|San Francisco Caltrain (Townsend at 4th)     |Temporary Transbay Terminal (Howard at Beale)|600     |
|Temporary Transbay Terminal (Howard at Beale)|Market at 10th            

**Найти количество велосипедов в системе**

In [33]:
num_bikes = df_trip.agg(F.countDistinct("bike_id").alias("num_bikes"))
num_bikes.show()

+---------+
|num_bikes|
+---------+
|      700|
+---------+



**Найти пользователей потративших на поездки более 3 часов**

In [34]:
users_time = (df_trip.groupBy("zip_code").agg((F.sum("duration")/3600).alias("total_hours")).filter(F.col("total_hours") > 3))

users_time.show()

+--------+------------------+
|zip_code|       total_hours|
+--------+------------------+
|   94102| 5313.339166666667|
|   95134|202.22861111111112|
|   84606|26.429166666666667|
|   80305|50.251666666666665|
|   60070| 8.033055555555556|
|   95519|            8.4175|
|   43085|3.2416666666666667|
|   91910|14.024444444444445|
|   77339|3.8091666666666666|
|   48063|3.8208333333333333|
|   95138| 43.15694444444444|
|   94610|1008.5077777777777|
|   94404| 997.0416666666666|
|   80301| 42.27472222222222|
|   91326|18.301388888888887|
|   90742|3.0458333333333334|
|   11205| 4.681111111111111|
|   95212| 3.216666666666667|
|   94309| 16.61277777777778|
|   22314| 8.671666666666667|
+--------+------------------+
only showing top 20 rows
